# Analítica Avanzada de Datos
## Práctica 2. Predicción de Gastos Médicos

# Equipo 9

Integrantes del equipo:
- Antonio Eugenio Daniel
- Domínguez Espinoza Juan Pablo
- Gutierrez Peña Montserrath
- León Villagomez Antonio

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Parte 1

## Carga y exploracion inicial

El dataset que utilizarán es **"US Health Insurance Dataset"**  https://www.kaggle.com/datasets/teertha/ushealthinsurancedataset

In [ ]:
data_raw = pd.read_csv('insurance.csv')
data_raw.head()

In [ ]:
data_raw.describe()

In [ ]:
data_raw.info()

Para poder realizar un analsis del dataset se busca identidicar el significado de las variables:
- age (edad)
    Representa la edad del cliente, es un dato de tipo entero.
- sex (sexo)
    Representa el sexo del cliente, es un dato de tipo cadena.
- bmi (imc/indice de masa coportal)
    Representa el indice de masa corporal del cliente, es un dato de tipo flotante.
- childen (hijos)
    Representa el numero de hijos del cliente, es un dato de tipo entero.
- smoker (fumador)
    Representa si el cliente es fumador o no, es un dato de tipo cadena(si o no).
- region
    Representa la parte especifica de residencia del cliente en Estados Unidos(norte, sur, este u oeste), es un dato de tipo cadena.
- charges (cargos)
    Representa los cargos medicos facturados del seguro del cliente, es un dato de tipo flotante.


Este dataset es utlizado con el proposito de carculo de costo para seguros, las variables presentan factores de interes para las aseguradoras al momento de determinvar la viabilidad y rentabilidad de un cliente segun las caracteristicas que posee.

El dataset posee 1338 registros, sin ningun valor nulo.

# Limpieza del data set

In [ ]:
# Verificar la existencia de valores nulos
data_raw.isnull().sum()

Se observa que no hay presencia de valores nulos en las dimensiones del dataset

In [ ]:
# Presencia de valores atipicos
sns.boxplot(x=data_raw['age'])
plt.show()

No se presentan valores atipicos, los limites son aproximadamente:
limite inferior: 18(aprox).
limite superior: 70(aprox).

In [ ]:
sns.boxplot(x=data_raw['bmi'])
plt.show()

En el indice de masa corporal se observa la presencia de valores atipicos, por lo que se usa el proceso de IQR.

In [ ]:
Q1 = data_raw["bmi"].quantile(0.25)
Q3 = data_raw["bmi"].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_bmi = data_raw[(data_raw["bmi"] < limite_inferior) | (data_raw["bmi"] > limite_superior)]

In [ ]:
outliers_bmi.sort_values(by="bmi", ascending=False)

Los valores atipicos encontrados no representan errores en la captura de los datos, en su lugar son casos extremos del indice de masa coorporal de los clientes, por lo que en este caso se decide preservarlos.

In [ ]:
sns.boxplot(x=data_raw['children'])
plt.show()

In [ ]:
sns.boxplot(x=data_raw['charges'])
plt.show()

In [ ]:
Q1 = data_raw["charges"].quantile(0.25)
Q3 = data_raw["charges"].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_charges = data_raw[(data_raw["charges"] < limite_inferior) | (data_raw["charges"] > limite_superior)]

In [ ]:
outliers_charges.sort_values(by="charges", ascending=False)

Por el volumen de los datos y que si presentan valores de costos para la aseguradora, se opta por el metodo de Winsorizing, en lugar de simplemente eliminarlos, ya que presentan el 10% de los datos.

In [ ]:
from scipy.stats.mstats import winsorize

data_raw['charges_winsorized'] = data_raw['charges'].clip(lower=limite_inferior, upper=limite_superior)

In [ ]:
sns.boxplot(x=data_raw['charges_winsorized'])
plt.show()

In [ ]:
data_clean = data_raw.copy()

data_clean.drop(columns=['charges'], inplace=True)

In [ ]:
data_clean.rename(columns={'charges_winsorized': 'charges'}, inplace=True)

In [ ]:
data_clean

## Observar las distribuciones de las clases

In [ ]:
data_clean.hist(bins=20, figsize=(15, 10), color='teal', edgecolor='black')
plt.tight_layout()
plt.show()